In [35]:
import os

from functools import partial
from typing import Callable

import evaluate

import numpy as np
import pandas as pd

from transformers import DataCollatorWithPadding, TrainingArguments, Trainer, pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel
from datasets import Dataset, ClassLabel
from evaluate import evaluator, combine

# Setup Data

In [36]:
dataset_path = "data/clean-toxic-tweets.csv"

dataset = pd.read_csv(filepath_or_buffer=dataset_path)

In [37]:
dataset.head(n=2)

,id,label,tweet,label_name,word_count,char_count,clean_text,clean_stemmed
0,1,0,@user when a father is dysfunctional and is s...,non-toxic,21,102,dysfunctional selfish drags dysfunction,dysfunct selfish drag dysfunct
1,2,0,@user @user thanks for #lyft credit i can't us...,non-toxic,22,122,credit cant cause dont offer wheelchair vans pdx,credit cant caus dont offer wheelchair van pdx


In [38]:
data = dataset[["clean_text", "label"]]
data.head(n=3)

,clean_text,label
0,dysfunctional selfish drags dysfunction,0
1,credit cant cause dont offer wheelchair vans pdx,0
2,majesty,0


In [39]:
data = data.rename({"clean_text": "text"}, axis=1)

In [40]:
# Labels are already 0 (non-toxic) and 1 (toxic) — no replacement needed
data["label"].value_counts()

label
0    29720
1     2242
Name: count, dtype: int64

In [41]:
data

,text,label
0,dysfunctional selfish drags dysfunction,0
1,credit cant cause dont offer wheelchair vans pdx,0
2,majesty,0
3,NaN,0
4,factsguide society,0
...,...,...
31957,ate isz youuu,0
31958,nina turner airwaves trying wrap mantle genuin...,0
31959,listening songs monday otw,0
31960,vandalised condemns act,1


In [42]:
data["label"].value_counts()

label
0    29720
1     2242
Name: count, dtype: int64

# Setup Model

In [43]:
model_name = "distilbert-base-uncased"

# Dataset

## Load Dataset

In [44]:
data = data.dropna(subset=["text"]).reset_index(drop=True)
dataset = Dataset.from_pandas(df=data)
dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 28862
})

## Labelling

In [45]:
classlabel = ClassLabel(num_classes=2, names=["non-toxic", "toxic"])
classlabel

ClassLabel(names=['non-toxic', 'toxic'])

In [46]:
dataset = dataset.cast_column(column="label", feature=classlabel)
dataset

Casting the dataset: 100%|██████████| 28862/28862 [00:00<00:00, 4590671.29 examples/s]


Dataset({
    features: ['text', 'label'],
    num_rows: 28862
})

In [47]:
dataset.features

{'text': Value('string'), 'label': ClassLabel(names=['non-toxic', 'toxic'])}

## Train Test Split

## Preprocess Data for Model

In [48]:
# Simple text preprocessing for the model
def preprocess_text(text):
    if not isinstance(text, str):
        return ""
    text = text.lower().strip()
    return text

In [49]:
sample = dataset[100]['text']
sample

'health benefits cucumbers'

In [50]:
preprocess_text(sample)

'health benefits cucumbers'

In [51]:
dataset = dataset.map(function=lambda x: {"text": preprocess_text(x)}, input_columns="text")

Map: 100%|██████████| 28862/28862 [00:00<00:00, 46110.39 examples/s]


In [52]:
dataset[0]["text"]

'dysfunctional selfish drags dysfunction'

## Train Test Split

In [53]:
dataset = dataset.train_test_split(test_size=0.2, stratify_by_column="label")

In [54]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 23089
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 5773
    })
})

## Tokenizer

In [55]:
tokenizer = AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

In [56]:
tokenizer

DistilBertTokenizerFast(name_or_path='distilbert-base-uncased', vocab_size=30522, model_max_length=512, is_fast=True, padding_side='right', truncation_side='right', special_tokens={'unk_token': '[UNK]', 'sep_token': '[SEP]', 'pad_token': '[PAD]', 'cls_token': '[CLS]', 'mask_token': '[MASK]'}, clean_up_tokenization_spaces=True),  added_tokens_decoder={
	0: AddedToken("[PAD]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	100: AddedToken("[UNK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	101: AddedToken("[CLS]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	102: AddedToken("[SEP]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	103: AddedToken("[MASK]", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
}

In [57]:
dataset = dataset.map(
    function=lambda x: tokenizer(x, truncation=True, max_length=18),
    input_columns="text",
)

Map: 100%|██████████| 5773/5773 [00:00<00:00, 18743.02 examples/s]


In [58]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 23089
    })
    test: Dataset({
        features: ['text', 'label', 'input_ids', 'attention_mask'],
        num_rows: 5773
    })
})

## Model

In [59]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2
)

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [60]:
# Use DataCollatorWithPadding to pad tokens and prepare batches, instead of doing it for the whole data, it is useful if you have a data doesn't fit into memory. 
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, max_length=18)

# Metrics

In [61]:
f1 = evaluate.load("f1")

In [62]:
def compute_metrics(eval_pred: np.ndarray, metric: evaluate.Metric):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels, average="binary")

#returns a new callable where some parameters of func are fixed. so you
compute_metrics_fn = partial(compute_metrics, metric=f1)

# Training

## Training Args

In [63]:
training_args = TrainingArguments(
    output_dir=os.path.join(os.curdir, "data"),
    overwrite_output_dir=True,
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_steps=50,
    logging_strategy="steps",
    logging_steps=50, 
    evaluation_strategy="steps",
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    load_best_model_at_end=True,
    save_steps=50,
    save_total_limit=1
)

/Users/anas/academy/worshop_3/.venv/lib/python3.12/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


## Trainer

In [64]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics_fn
)

  2%|▏         | 650/31970 [21:44<17:27:30,  2.01s/it]



## Train Model

In [65]:
trainer.train()

  0%|          | 0/28870 [00:00<?, ?it/s]/Users/anas/academy/worshop_3/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/anas/academy/worshop_3/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
/Users/anas/academy/worshop_3/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/anas/academy/worshop_3/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation 

{'loss': 0.3377, 'grad_norm': 2.9150190353393555, 'learning_rate': 1.996536196744025e-05, 'epoch': 0.02}


  0%|          | 50/28870 [00:29<1:18:52,  6.09it/s]

{'eval_loss': 0.30272138118743896, 'eval_f1': 0.0, 'eval_runtime': 13.2046, 'eval_samples_per_second': 437.197, 'eval_steps_per_second': 54.678, 'epoch': 0.02}


/Users/anas/academy/worshop_3/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2717: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(
  0%|          | 97/28870 [00:38<1:20:13,  5.98it/s] 

KeyboardInterrupt: 

## Evaluate Model

In [ ]:
results = trainer.evaluate()
results

: 

: 

In [ ]:
# Get predictions on the test set
predictions = trainer.predict(dataset["test"])

preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

# Classification report
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(labels, preds, target_names=["non-toxic", "toxic"]))

In [ ]:
# Confusion matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(labels, preds)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=["non-toxic", "toxic"], 
            yticklabels=["non-toxic", "toxic"], ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")
plt.show()